# Processing MESACLIP data - atmosphere monthly, single ensembles

* **Description**: Reads in MESACLIP historical and crops to just the Arctic Amplification/temperature
* **Creator**: Alice DuVivier
* **Date**: December 2025

In [ ]:
import xarray as xr
import numpy as np
import pandas as pd
from glob import glob
from datetime import datetime
import xesmf as xe
import matplotlib.pyplot as plt
import scipy.sparse as sps

import dask
from distributed import Client
from ncar_jobqueue import NCARCluster

## Spin up DASK cluster

In [ ]:
# Create our NCAR Cluster - which uses PBSCluster under the hood
cluster = NCARCluster(walltime='2:00:00', cores=1, processes=1, memory='100 GB', 
                      resource_spec='select=1:ncpus=1:mem=100GB',interface='ext')

# Spin up 32 workers
cluster.scale(32)

# Assign the cluster to our Client
client = Client(cluster)

In [ ]:
client
# cluster.close()

## Set MESACLIP ensemble information

In [ ]:
# RCP 6.0
path_base = '/gdex/data/d651008/'
case_base = 'b.e13.BRCP60C5.ne120_t12.cesm-ihesp-hires1.0.'
# list 10 members of MESACLIP - last one is the first and named differently
ens_mems = ['44-2006-2100.001', '44-2006-2100.002', 
            '44-2006-2100.003', '46-2006-2100.004', 
            '44-2006-2100.005', '46-2006-2100.006', 
            '46-2006-2100.007', '46-2006-2100.008', 
            '46-2006-2100.009', '46-2006-2100.010'] 
nens = len(ens_mems)

## Load in ATMOSPHERIC Data

In [ ]:
# Choose variable to load

var_in = 'TREFHT'

In [ ]:
# Choose ensemble to load
n = 9
ens_in = ens_mems[n]
print(ens_in)

In [ ]:
%%time

print('loading data for member: '+ens_in)

# set case
case = case_base + ens_in
# set data directory
path = path_base+case+'/atm/proc/tseries/month_1'
# get list of files
files = sorted(glob(f'{path}/{case}.cam.h0.{var_in}.??????-??????.nc'))       
# load files
ds_tmp = xr.open_mfdataset(files, data_vars="minimal", coords='minimal', compat="override", parallel=True, 
                           concat_dim="time", combine='nested', decode_times=True)
# take an average over the time bounds to get the right time dimension
ds_tmp["time"] = ds_tmp.time_bnds.compute().mean(dim="nbnd")


In [ ]:
# check data
ds_tmp.TREFHT.isel(time=100).plot()

### Remap to a regular grid

In [ ]:
#wgtfile = '/glade/work/fredc/Regrid_Maps/ne120np4_pentagons_to_latlon_0.25x0.25_0E_bilinear_20210903.nc'
wgtfile = '/glade/work/fredc/Regrid_Maps/ne120np4_pentagons_to_latlon_0.25x0.25_0E_conserve_20200309.nc'
dsw = xr.open_dataset(wgtfile)

In [ ]:
def remap_camse(ds, dsw, varlst=[]):
    dso = xr.full_like(ds.drop_dims('ncol'), np.nan)
    lonb = dsw.xc_b.values.reshape([dsw.dst_grid_dims[1].values, dsw.dst_grid_dims[0].values])
    latb = dsw.yc_b.values.reshape([dsw.dst_grid_dims[1].values, dsw.dst_grid_dims[0].values])
    weights = sps.coo_matrix((dsw.S, (dsw.row-1, dsw.col-1)), shape=[dsw.sizes['n_b'], dsw.sizes['n_a']])
    if not varlst:
        for varname in list(ds):
            if 'ncol' in(ds[varname].dims):
                varlst.append(varname)
        if 'lon' in varlst: varlst.remove('lon')
        if 'lat' in varlst: varlst.remove('lat')
        if 'area' in varlst: varlst.remove('area')
    for varname in varlst:
        shape = ds[varname].shape
        invar_flat = ds[varname].values.reshape(-1, shape[-1])
        remapped_flat = weights.dot(invar_flat.T).T
        remapped = remapped_flat.reshape([*shape[0:-1], dsw.dst_grid_dims[1].values, dsw.dst_grid_dims[0].values])
        dimlst = list(ds[varname].dims[0:-1])
        dims={}
        coords={}
        for it in dimlst:
            dims[it] = dso.sizes[it]
            coords[it] = dso.coords[it]
        dims['lat'] = int(dsw.dst_grid_dims[1])
        dims['lon'] = int(dsw.dst_grid_dims[0])
        coords['lat'] = latb[:,0]
        coords['lon'] = lonb[0,:]
        remapped = xr.DataArray(remapped, coords=coords, dims=dims, attrs=ds[varname].attrs)
        dso = xr.merge([dso, remapped.to_dataset(name=varname)])
    varlst.clear() #not sure why this is needed but varlst was not cleared and it created issues with successive call to remap_camse
    return dso

In [ ]:
%%time
ds = remap_camse(ds_tmp,dsw)

In [ ]:
# check data
ds.TREFHT.isel(time=100).plot()

## Subset file to Arctic Only
- ">50N"

In [ ]:
ds.lat[560:720]

In [ ]:
ds_sub = ds.isel(lat=slice(560,720))

In [ ]:
# check data
ds_sub.TREFHT.isel(time=100).plot()

## Export and save the file
- NOTE: YOU MUST MANUALLY ENTER THE VARIABLE NAME BELOW

In [ ]:
# set output path and names
path_out = '/glade/campaign/cgd/ppc/duvivier/arctic_actionable/DATA/mesaclip_all_arctic/monthly/'

# set output file name
nn = n+1
if nn < 10:
    ensnum = '00'+str(nn)
elif nn >= 10:
    ensnum = '010'

file_out = case_base+ensnum+'_2015-2099_'+var_in+'_arctic_monthly.nc'
fout = path_out + file_out
print(fout)

In [ ]:
ds_out = xr.Dataset()

ds_out['TREFHT'] = ds_sub.TREFHT

# change the attributes
ds_out.attrs['author'] = 'Alice DuVivier'
ds_out.attrs['date_processed'] = datetime.now().strftime('%Y-%m-%d')
ds_out.attrs['contents'] = f'Subset of RCP6.0 MESACLIP Data'

ds_out

In [ ]:
# Print the dimensions
print("Dimensions:")
for dim in ds_out.dims:
    print(f"\t{dim}: {ds_out[dim].values.shape}")

# Print the coordinates
print("Coordinates:")
for coord in ds_out.coords:
    print(f"\t{coord}:")
    print(f"\t\t{ds_out.coords[coord].values}")
    
# Print the attributes
print("Attributes:")
for attr in ds_out.attrs:
    print(f"\t{attr}: {ds_out.attrs[attr]}")

In [ ]:
# calculate the size of the dataset in GB
size_gb = ds_out.nbytes / (1024**3)
print(f"The dataset is approximately {size_gb:.2f} GB.")

In [ ]:
# Export the dataset to NetCDF with all attributes and coordinates
ds_out.to_netcdf(fout)

In [ ]:
#cluster.close()